# Xente Transaction EDA — Credit Risk Proxy

Exploratory analysis for RFM-based proxy target and feature engineering.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RAW_PATH = Path("../data/raw/data.csv")
if not RAW_PATH.exists():
    raise FileNotFoundError("Place data.csv in data/raw/")

df = pd.read_csv(RAW_PATH)
df["TransactionStartTime"] = pd.to_datetime(df["TransactionStartTime"])
df.head()

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15 02:18:49+00:00,2,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15 02:19:08+00:00,2,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15 02:44:21+00:00,2,0
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15 03:32:55+00:00,2,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15 03:34:21+00:00,2,0


In [3]:
print(f"Transactions: {len(df):,}")
print(f"Customers: {df['CustomerId'].nunique():,}")
print(f"Fraud rate: {df['FraudResult'].mean():.4f}")
df.describe(include='all').T.head(15)

Transactions: 95,662
Customers: 3,742
Fraud rate: 0.0020


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
TransactionId,95662,95662,TransactionId_76871,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BatchId,95662,94809,BatchId_67019,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AccountId,95662,3633,AccountId_4841,30893,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SubscriptionId,95662,3627,SubscriptionId_3829,32630,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CustomerId,95662,3742,CustomerId_7343,4091,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CurrencyCode,95662,1,UGX,95662,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CountryCode,95662.0,NaN,NaN,NaN,256.0,256.0,256.0,256.0,256.0,256.0,0.0
ProviderId,95662,6,ProviderId_4,38189,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ProductId,95662,23,ProductId_6,32635,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ProductCategory,95662,9,financial_services,45405,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
import sys
sys.path.insert(0, "..")
from src.data_processing import build_modeling_dataset, compute_weight_of_evidence

dataset, metadata = build_modeling_dataset(df)
print("Default proxy rate:", metadata["default_rate"])
pd.DataFrame(metadata["iv_summary"]).head(10)

Default proxy rate: 0.3810796365579904


,feature,iv
0,recency_days,31.551909
1,tenure_days,10.256827
2,transactions_per_day,7.192165
3,frequency,0.874047
4,debit_share,0.630768
5,monetary_total,0.600527
6,transaction_value_std,0.389731
7,max_transaction_value,0.248937
8,product_category_diversity,0.230869
9,avg_transaction_value,0.178036


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col in zip(axes, ["recency_days", "frequency", "monetary_total"]):
    dataset[col].hist(ax=ax, bins=30, edgecolor="k")
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
corr = dataset[[c for c in metadata["selected_features"] if c in dataset.columns] + ["is_high_risk"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature correlation with proxy default")
plt.tight_layout()
plt.show()

## Top 3–5 Insights

- **RFM distributions are highly skewed:** `monetary_total` and `frequency` show long tails, indicating a small fraction of customers account for large transaction volumes.
- **Recency separates disengaged customers:** high `recency_days` aligns with the proxy `is_high_risk` cluster, supporting the RFM-based proxy design.
- **Few strong predictors by IV:** Information Value ranking (metadata) identifies a compact set of features with predictive signal — useful for parsimonious, interpretable models under Basel II.
- **Class imbalance exists:** the proxy `is_high_risk` class is a minority, so use appropriate evaluation (ROC-AUC, precision-recall) and sampling strategies during training.

These insights guided feature selection and the choice of an interpretable baseline model.